# Dataset C – COVID-19 Vaccine Uptake Hackathon
**Team:** _add names_  |  **Target:** `covid_vaccine` (1 = vaccinated, 0 = not vaccinated)

**Workflow:** 1) Setup → 2) EDA → 3) Metrics → 4) Split → 5) Pre-processing → 6) Model comparison → 7) Tuning → 8) Hold-out → 9) Submission

## 1. Setup

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import cross_validate
from sklearn.metrics import (roc_auc_score, accuracy_score, balanced_accuracy_score, f1_score,
                             precision_score, recall_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay)
from scipy.stats import randint, uniform, loguniform

RANDOM_STATE = 42
sns.set_style("whitegrid")

# Colab: upload the CSV to the session (left sidebar -> Files) or mount Drive.
DATA_PATH = "dataset_C_training.csv"
df = pd.read_csv(DATA_PATH)
TARGET = "covid_vaccine"
print(df.shape)
df.head()

## 2. Exploratory Data Analysis

In [ ]:
df.info()
df.describe().T

### 2.1 Target balance

In [ ]:
print(df[TARGET].value_counts(normalize=True).round(3))
df[TARGET].value_counts().plot(kind="bar", title="Target distribution (0 = not vaccinated, 1 = vaccinated)")
plt.show()

**Observation:** roughly one third of respondents are vaccinated, so the classes are imbalanced. Accuracy would be a misleading metric, and the costs below make that even more important.

### 2.2 Missing values

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]
miss.plot(kind="barh", figsize=(7, 5), title="Share of missing values per column")
plt.show()
miss.round(3)

**Observation:** `employment_sector` (~49%) and `health_insurance` (~40%) have a lot of missing values, and `income_poverty` ~16%. The missingness may itself be informative (e.g. not employed → no sector), so we keep a "Missing" category / missing indicator instead of dropping rows. Compare the target rate for missing vs. non-missing below.

In [ ]:
for c in ["employment_sector", "health_insurance", "income_poverty"]:
    print(c)
    print(df.groupby(df[c].isna())[TARGET].mean().rename({False: "present", True: "missing"}).round(3), "\n")

### 2.3 Feature distributions

In [ ]:
num_cols = df.select_dtypes("number").columns.drop(["respondent_id", TARGET]).tolist()
cat_cols = df.select_dtypes(exclude="number").columns.tolist()

fig, axes = plt.subplots(6, 4, figsize=(18, 20))
for ax, c in zip(axes.ravel(), num_cols):
    df[c].value_counts(dropna=False).sort_index().plot(kind="bar", ax=ax, title=c)
    ax.set_xlabel("")
for ax in axes.ravel()[len(num_cols):]: ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
for ax, c in zip(axes.ravel(), cat_cols):
    df[c].fillna("Missing").value_counts().plot(kind="barh", ax=ax, title=c)
for ax in axes.ravel()[len(cat_cols):]: ax.axis("off")
plt.tight_layout(); plt.show()

### 2.4 Relationship with the target

In [ ]:
# Vaccination rate per category / value (top drivers)
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
for ax, c in zip(axes.ravel(), cat_cols):
    df.assign(**{c: df[c].fillna("Missing")}).groupby(c)[TARGET].mean().sort_values().plot(kind="barh", ax=ax, title=f"Vaccination rate by {c}")
    ax.set_xlim(0, 1)
for ax in axes.ravel()[len(cat_cols):]: ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
key = ["doctor_recc_covid", "opinion_covid_vacc_effective", "opinion_covid_risk",
       "opinion_covid_sick_from_vacc", "covid_concern", "health_worker", "health_insurance",
       "number_chronic_med_conditions"]
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, c in zip(axes.ravel(), key):
    df.groupby(c)[TARGET].mean().plot(kind="bar", ax=ax, title=f"Vaccination rate by {c}")
    ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

### 2.5 Correlations

In [ ]:
corr = df[num_cols + [TARGET]].corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Correlation matrix (numeric features + target)")
plt.show()
corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).head(10)

**EDA take-aways (edit after you've read the plots):**
- Doctor recommendation, belief in vaccine effectiveness, and perceived risk are expected to be among the strongest signals.
- Age group, health insurance and health-worker status also show clear differences in vaccination rate.
- Behavioural features are correlated with each other, which matters for linear models but not much for tree models.

## 3. Evaluation metrics
The classes are imbalanced (~33% vaccinated), so accuracy alone is misleading. We report:
- **ROC-AUC** (main metric for model selection and tuning – threshold-independent)
- **F1** (balance of precision and recall for the vaccinated class), **balanced accuracy**, precision, recall and accuracy

Baseline: always predicting "not vaccinated" gives ~67% accuracy but F1 = 0.

In [ ]:
SCORING = {"roc_auc": "roc_auc", "f1": "f1", "balanced_acc": "balanced_accuracy",
           "precision": "precision", "recall": "recall", "accuracy": "accuracy"}
print("Majority-class baseline accuracy:", round(1 - df[TARGET].mean(), 3))

## 4. Train / hold-out split

In [ ]:
X = df.drop(columns=[TARGET, "respondent_id"])   # id is random -> no information
y = df[TARGET]

X_train, X_hold, y_train, y_hold = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(X_train.shape, X_hold.shape)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 5. Pre-processing
Two options are compared:
- **Option A:** median imputation (numeric), "Missing" category (categorical), one-hot encoding, scaling for linear models only.
- **Option B:** Option A + missing-value indicators for numeric columns (e.g. `health_insurance` has ~40% missing).

All steps are inside the pipeline, so they are fitted on training folds only (no leakage).

In [ ]:
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()

def make_preprocessor(add_indicator=False, scale=False):
    num_steps = [("imp", SimpleImputer(strategy="median", add_indicator=add_indicator))]
    if scale: num_steps.append(("sc", StandardScaler()))
    cat_steps = [("imp", SimpleImputer(strategy="constant", fill_value="Missing")),
                 ("oh", OneHotEncoder(handle_unknown="ignore"))]
    return ColumnTransformer([("num", Pipeline(num_steps), num_cols),
                              ("cat", Pipeline(cat_steps), cat_cols)])

def cv_report(name, pipe):
    """5-fold CV: mean and std of each metric."""
    s = cross_validate(pipe, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1)
    row = {"model": name}
    for k in SCORING:
        row[k] = s[f"test_{k}"].mean()
        row[k + "_std"] = s[f"test_{k}"].std()
    return row

## 6. Model comparison (default hyper-parameters, 5-fold CV)

In [ ]:
candidates = {
 "LogReg (A)":       Pipeline([("pre", make_preprocessor(False, True)),  ("m", LogisticRegression(max_iter=2000))]),
 "LogReg (B)":       Pipeline([("pre", make_preprocessor(True,  True)),  ("m", LogisticRegression(max_iter=2000))]),
 "RandomForest (A)": Pipeline([("pre", make_preprocessor(False, False)), ("m", RandomForestClassifier(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=RANDOM_STATE))]),
 "RandomForest (B)": Pipeline([("pre", make_preprocessor(True,  False)), ("m", RandomForestClassifier(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=RANDOM_STATE))]),
 "HistGB (A)":       Pipeline([("pre", make_preprocessor(False, False)), ("m", HistGradientBoostingClassifier(random_state=RANDOM_STATE))]),
 "HistGB (B)":       Pipeline([("pre", make_preprocessor(True,  False)), ("m", HistGradientBoostingClassifier(random_state=RANDOM_STATE))]),
}
results = pd.DataFrame([cv_report(n, p) for n, p in candidates.items()]).sort_values("roc_auc", ascending=False).reset_index(drop=True)
results[["model", "roc_auc", "roc_auc_std", "f1", "f1_std", "balanced_acc", "precision", "recall", "accuracy"]].round(3)

**Reading the table:** `*_std` is the spread across the 5 folds. If two models differ by less than that spread, treat them as equivalent. Compare option A vs B to say whether the missing indicators helped.

## 7. Hyper-parameter tuning (scored on ROC-AUC)

In [ ]:
tune_spaces = {
 "LogReg": (Pipeline([("pre", make_preprocessor(True, True)), ("m", LogisticRegression(max_iter=3000))]),
            {"m__C": loguniform(1e-3, 1e2), "m__penalty": ["l1", "l2"], "m__solver": ["liblinear"]}),
 "RandomForest": (Pipeline([("pre", make_preprocessor(True, False)), ("m", RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE))]),
            {"m__n_estimators": randint(200, 600), "m__max_depth": [None, 6, 10, 16],
             "m__min_samples_leaf": randint(1, 20), "m__max_features": ["sqrt", 0.3, 0.5]}),
 "HistGB": (Pipeline([("pre", make_preprocessor(True, False)), ("m", HistGradientBoostingClassifier(random_state=RANDOM_STATE))]),
            {"m__learning_rate": loguniform(0.01, 0.3), "m__max_iter": randint(100, 500),
             "m__max_depth": [None, 3, 5, 8], "m__min_samples_leaf": randint(10, 80),
             "m__l2_regularization": loguniform(1e-3, 10)}),
}
tuned = {}
for name, (pipe, space) in tune_spaces.items():
    search = RandomizedSearchCV(pipe, space, n_iter=20, scoring="roc_auc", cv=cv,
                                random_state=RANDOM_STATE, n_jobs=-1, refit=True)
    search.fit(X_train, y_train)
    tuned[name] = search.best_estimator_
    print(f"{name:13s} best CV ROC-AUC = {search.best_score_:.4f}   params = {search.best_params_}")

In [ ]:
tuned_results = pd.DataFrame([cv_report(f"{n} (tuned)", est) for n, est in tuned.items()]).sort_values("roc_auc", ascending=False).reset_index(drop=True)
tuned_results[["model", "roc_auc", "roc_auc_std", "f1", "f1_std", "balanced_acc", "precision", "recall", "accuracy"]].round(3)

## 8. Hold-out validation
The hold-out set was never used for tuning. Predictions use the default 0.5 threshold.

In [ ]:
hold_rows, hold_proba = [], {}
for name, est in tuned.items():
    est.fit(X_train, y_train)
    p = est.predict_proba(X_hold)[:, 1]; pred = (p >= 0.5).astype(int)
    hold_proba[name] = p
    hold_rows.append(dict(model=name, roc_auc=roc_auc_score(y_hold, p), f1=f1_score(y_hold, pred),
                          balanced_acc=balanced_accuracy_score(y_hold, pred),
                          precision=precision_score(y_hold, pred), recall=recall_score(y_hold, pred),
                          accuracy=accuracy_score(y_hold, pred)))
holdout = pd.DataFrame(hold_rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)
holdout.round(3)

In [ ]:
# ROC curves + confusion matrix of the best model
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for name, p in hold_proba.items():
    RocCurveDisplay.from_predictions(y_hold, p, name=name, ax=ax[0])
ax[0].plot([0, 1], [0, 1], "k--"); ax[0].set_title("ROC curves (hold-out)")
best = holdout.loc[0, "model"]
ConfusionMatrixDisplay.from_predictions(y_hold, (hold_proba[best] >= 0.5).astype(int), ax=ax[1], colorbar=False)
ax[1].set_title(f"Confusion matrix – {best}")
plt.show()
print(classification_report(y_hold, (hold_proba[best] >= 0.5).astype(int), target_names=["not vaccinated", "vaccinated"]))

In [ ]:
# Which features matter? (permutation importance on the best tree model, hold-out data)
from sklearn.inspection import permutation_importance
m = tuned["RandomForest"]
imp = permutation_importance(m, X_hold, y_hold, scoring="roc_auc", n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
pd.Series(imp.importances_mean, index=X_hold.columns).sort_values().tail(12).plot(kind="barh", title="Permutation importance (ROC-AUC drop)")
plt.show()

**Critical evaluation (fill in):** compare CV vs hold-out scores (a big gap = over-fitting), mention the fold spread, say which pre-processing option helped, and justify the final ranking.

## 9. Submission
Re-fit on **all** labelled data, predict the hidden test set, save one file per model ranked best to worst.
⚠️ Adjust file/column names to the format given with the test set.

In [ ]:
TEST_PATH = "dataset_C_test.csv"                      # <- the test FEATURES file (same columns as training, no covid_vaccine)
EXAMPLE_PATH = "dataset_C_example_submission.csv"     # format reference: respondent_id, covid_vaccine

test = pd.read_csv(TEST_PATH)
example = pd.read_csv(EXAMPLE_PATH)
X_test = test.drop(columns=["respondent_id"], errors="ignore")[X.columns]

# sanity checks: same ids as the example submission
assert set(test["respondent_id"]) == set(example["respondent_id"]), "test ids differ from example submission"

for rank, row in holdout.iterrows():            # best first
    name = row["model"]
    est = tuned[name]
    est.fit(X, y)                                # re-fit on all labelled data
    pred = est.predict(X_test).astype(float)     # 0.0 / 1.0, same as the example file
    sub = pd.DataFrame({"respondent_id": test["respondent_id"], TARGET: pred})
    sub = sub.set_index("respondent_id").loc[example["respondent_id"]].reset_index()   # same row order as example
    assert list(sub.columns) == list(example.columns) and len(sub) == len(example)
    sub.to_csv(f"prediction_{rank+1}_{name}.csv", index=False)
    print(rank + 1, name, "-> share predicted vaccinated:", round(pred.mean(), 3))